
Project : Pentaho Log Intelligence

Layer   : Silver

Notebook: 03_Silver_Transformation_PentahoLog

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Silver Delta table.

Author: Ernesto Felipe Garay Cervantes



#### Recibimiento de Parametros 

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")

archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))

print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo)

#### Configuracion

In [0]:
from pyspark.sql.functions import (
    col,
    regexp_extract,
    current_timestamp
)

In [0]:
CATALOG = "pentaho_logs"

BRONZE_TABLE_PENTAHO = "pentaho_logs.bronze.bronze_logs_pentaho"

SILVER_TABLE_PENTAHO= "pentaho_logs.silver.silver_logs_pentaho"

#### Lectura  de tabla Bronze 

In [0]:
df_pentaho_sl = spark.table(BRONZE_TABLE_PENTAHO).filter(col("file_name").isin(archivos_nuevos))

#display(df_pentaho_sl.limit(20))

In [0]:
df_pentaho_sl.printSchema()

#### Enriquecimiento Data Frame

In [0]:
from pyspark.sql.functions import (col,regexp_extract,to_timestamp)

df_silver_pentaho = (df_pentaho_sl.withColumn("mensaje_pentaho_log",regexp_extract(col("descripcion"), r"^\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2},\d{3}\s+\w+\s+(.*)",1))
                                  .withColumn("proceso_pentaho_log",regexp_extract(col("descripcion"),r"\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}\.\d{3}\s+-\s+(.*?)\s+-",1))
        
                   )
#display(display(df_silver_pentaho))


In [0]:
from pyspark.sql.functions import col, trim, when

df_pentaho = (
    spark.table("pentaho_logs.gold.gold_logs_pentaho")
    
    .withColumn(
        "fecha",
        when(trim(col("fecha")) == "", None)
        .otherwise(col("fecha"))
    )
    
    .withColumn(
        "hora",
        when(trim(col("hora")) == "", None)
        .otherwise(col("hora"))
    )
    
    .withColumn(
        "Nivel",
        when(trim(col("Nivel")) == "", None)
        .otherwise(col("Nivel"))
    )
)

In [0]:
from pyspark.sql.functions import to_date

df_pentaho = df_pentaho.withColumn(
    "fecha",
    to_date(col("fecha"), "yyyy-MM-dd")
)


#### validación DATAFRAME

In [0]:
# Número de registros
print(f"Total de líneas: {df_silver_pentaho.count():,}")

# Estructura
df_silver_pentaho.printSchema()

In [0]:
from pyspark.sql.functions import col, sum, when

df_silver_pentaho.select(
    sum(when(col("mensaje_pentaho_log").isNull(), 1).otherwise(0)).alias("mensaje_pentaho_log_null"),
    sum(when(col("proceso_pentaho_log").isNull(), 1).otherwise(0)).alias("proceso_pentaho_log_null")
).show()


#### Creación Tabla Silver Pentaho_Log

In [0]:
SILVER_TABLE_PENTAHO = "pentaho_logs.silver.silver_logs_pentaho"
(
    df_silver_pentaho.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE_PENTAHO)
)

In [0]:
#display(spark.table(SILVER_TABLE_PENTAHO).limit(20))